In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Load dataset with 20 broad categories
categories = [
    "sci.space", "sci.electronics", "sci.med", "comp.graphics", "comp.os.ms-windows.misc",
    "rec.sport.baseball", "rec.sport.hockey", "talk.politics.misc", "talk.religion.misc",
    "misc.forsale"
]
newsgroups = fetch_20newsgroups(subset="all", categories=categories, remove=("headers", "footers", "quotes"))

# Convert text to TF-IDF feature vectors
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)  # Limit feature size for speed
X = vectorizer.fit_transform(newsgroups.data)
y = newsgroups.target  # Category labels

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Train the Logistic Regression classifier
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

# Evaluate on test set
y_pred = logreg.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))


Logistic Regression Accuracy: 0.8073196986006459


In [3]:
from sklearn.naive_bayes import MultinomialNB

# Train the Naïve Bayes classifier
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Evaluate on test set
y_pred_nb = nb.predict(X_test)
print("Naïve Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))


Naïve Bayes Accuracy: 0.8089343379978472


In [4]:
def classify_text(text, model):
    X_new = vectorizer.transform([text])  # Convert text to numerical features
    predicted_category = newsgroups.target_names[model.predict(X_new)[0]]
    return predicted_category

sample_text = "The recent discovery of a new exoplanet has excited many astronomers."
print("Predicted Category (Logistic Regression):", classify_text(sample_text, logreg))
print("Predicted Category (Naïve Bayes):", classify_text(sample_text, nb))


Predicted Category (Logistic Regression): sci.space
Predicted Category (Naïve Bayes): sci.space


In [5]:
import os
import re

# Cargar modelo de spaCy en español y stopwords
import spacy
nlp = spacy.load("es_core_news_sm")

# import nltk
from nltk.corpus import stopwords
spanish_stopwords = set(stopwords.words("spanish"))

# Precompilar la regex para limpiar el texto
CLEAN_PATTERN = re.compile(r"<.*?>|[^a-zA-ZáéíóúüñÁÉÍÓÚÜÑ\s]")
CHUNKS_DIR = '/home/matias/RAG_master/storage/chunks'

def clean_text_batch(texts: list[str]) -> list[str]:
    """
    Limpia y lematiza un lote de textos utilizando spaCy en modo pipe.
    """
    cleaned_texts = []
    # Preprocesar cada texto con regex
    preprocessed_texts = [CLEAN_PATTERN.sub(" ", text).lower().strip() for text in texts]
    
    # Procesar el lote con nlp.pipe (esto es más rápido)
    for doc in nlp.pipe(preprocessed_texts, batch_size=32):
        # Filtrar stopwords y obtener lemas en un solo paso
        tokens = [token.lemma_ for token in doc if token.text not in spanish_stopwords]
        cleaned_texts.append(" ".join(tokens))
    return cleaned_texts

# %%
# Cargar textos de chunks en una lista (en vez de procesarlos uno por uno)
def load_all_chunk_texts(chunk_ids: list[str]) -> list[tuple[str, str]]:

    """
    Carga todos los textos de chunks dados sus IDs. 
    Se asume que los textos se encuentran en './chunks/<chunk_id>.txt'.
    Retorna una lista de tuplas (chunk_id, raw_text).
    """

    texts = []
    for chunk_id in chunk_ids:
        file_path = os.path.join(CHUNKS_DIR, f"{chunk_id}.txt")
        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                raw_text = f.read()
                if raw_text.strip():
                    texts.append((chunk_id, raw_text))
    return texts


In [6]:
import pandas as pd
from chunk_query import ChunkQuery  # Tu clase de consulta, tal como la definimos antes

CHUNKS_METADATA_FILE= "/home/matias/RAG_master/storage/tables/chunks_index.json"

df_metadata = pd.read_json(CHUNKS_METADATA_FILE, orient="index")

# 2. Inicializar la clase de consulta y filtrar los chunks de la carpeta /Academic/
cq = ChunkQuery(df_metadata)


# Ejemplo de query: Filtrar aquellos chunks cuyo 'original_path' contenga '/Academic/'
academic_chunk_ids = cq.query_custom("original_path.str.contains('/Academic/1_Input_Raw/Drafts_Papers_Early_Drafts/2024_1/AGGVOL')")
print("Cantidad de chunks en /Academic/1_Input_Raw/Drafts_Papers_Early_Drafts:", len(academic_chunk_ids))

# Cargar todos los textos
all_chunks = load_all_chunk_texts(academic_chunk_ids[:100])  # academic_chunk_ids es la lista filtrada
chunk_ids, raw_texts = zip(*all_chunks)  # Separar IDs y textos
chunk_ids, cleaned_texts = chunk_ids, clean_text_batch(raw_texts[:100])  # Separar IDs y textos


FileNotFoundError: File /home/matias/RAG_master/storage/tables/chunks_index.json does not exist